## Extract Segments 

Script for extracting labels for segmentation based CNN

## Importing Libraries

In [1]:
import os.path as op
import mne 
import os
import copy
import numpy as np 
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
matplotlib.use('QtAgg') 
import h5py
import tensorflow as tf

mne.set_log_level('CRITICAL')  # or 'ERROR' or False

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

## Functions

In [2]:
def apply_segments(C, B,m,S):
    """
    Replace values in C with 75 for each pair of indices in B.
    B must have an even number of elements: [start1, end1, start2, end2, ...]
    m is the muscle type that corresponds to contractions
    """
    i = 0 
    Z = copy.deepcopy(C)
    if B == [0,0]:
        return C,Z
    
    B = np.array(B, dtype=int)
    pairs = B.reshape(-1, 2)  # shape (n_pairs, 2)

    for start, end in pairs:
        type = m[i]
        if (int(type) == 1):
            C[start - S : end - S] = 75
        elif (int(type) == 2):
            Z[start - S : end - S] = 75
        i += 1 
    
    return C,Z

In [3]:
def extract_data(refs, f):
    cells = []
    for ref in refs:
        obj = f[ref]
        if obj.shape == (0, 0):
            cells.append(None)
        else:
            arr = np.array(obj).T
            cells.append(arr)
    return cells

def extract_strings(refs, f):
    cells = []
    for ref in refs:
        obj = f[ref]
        if obj.shape == (0, 0):
            cells.append(None)
        else:
            s = ''.join(chr(c) for c in obj[:].flatten())
            cells.append(s)
    return cells



In [4]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjects(subject_name,raw_path):
    global frq

   # subject_name = subject + block 
    file= op.join(raw_path,subject_name)
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    # raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return df_triggers, raw 

## Loading Data

In [5]:
with h5py.File('cell_data_all.mat', 'r') as f:
    refs = f['data']
    
    subj        = extract_strings(refs[0, :].flatten(), f)
    contractions = extract_data(refs[1, :].flatten(), f)
    muscle_type  = extract_data(refs[2, :].flatten(), f)



In [6]:
subject_exclude = ["NL01SS", "NL02IF",
                   "NL05WW01", "RL12JL03", "RL07BR02", "RL22AC05",
                   "RL16CM", "RL11JH","RL18CL", "RL21AC","RL03JG"] 
df = pd.DataFrame({
    'subj': subj,
    'contractions': contractions,
    'muscle_type': muscle_type
})

df = df[~df['subj'].isin(subject_exclude)].reset_index(drop=True)

In [7]:
df_trials = pd.read_csv('/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Trial_information_narcolepsy.csv')



df_trials_filtered = df_trials[df_trials['Presentation'] == 1][['Subject', 'Nap_ID']]
df_trials_filtered['contraction']=contractions
df_trials_filtered['muscle_type']=muscle_type

In [8]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']
subject_exclude = ["NL01SS", "NL02IF",
                   "NL05WW01", "RL12JL03", "RL07BR02", "RL22AC05",
                   "RL16CM", "RL11JH","RL18CL", "RL21AC","RL12JL","RL03JG05"]  # trials to exclude based on trigger count 


inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/full_EEG_dataset/"
df_trials = pd.read_csv('/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Trial_information_narcolepsy.csv')

current_index=0
inter_trigger_length=10
window = 50 
step = 1 

In [9]:
segments = pd.DataFrame(
        index=range(num_epochs),
        columns=["Subject", "Nap_ID","Trigger", "Contraction Corr", "Contraction Zygo"])   

i = 0 
for filename in os.listdir(raw_path):
    if filename.endswith(".edf"): 
        
        block = filename.split(".")[0][-1] 
        subject = filename.split(".")[0][:-2]

        if subject not in subject_exclude and filename.split(".")[0] not in subject_exclude:
            df_triggers,raw  = pre_process_subjects(filename,raw_path)

            events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
            events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest

            # filter df_triggers 
            df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
            df_triggers.drop(columns=['dunno'],inplace=True)
            df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']

            df_trials_filtered_subj = df_trials_filtered[df_trials_filtered['Subject'] == subject]
            df_trials_filtered_subj = df_trials_filtered_subj[df_trials_filtered['Nap_ID'] == int(block)]


            df_triggers['contraction']=df_trials_filtered_subj['contraction'].values
            df_triggers['contraction'] = df_triggers['contraction'].apply(lambda x: x.T.flatten().tolist())
            df_triggers['muscle_type']=df_trials_filtered_subj['muscle_type'].values
            # loop over triggers data frame for each subject
            for t in range(len(df_triggers)):
                lenght_epoch_zygo = np.zeros(int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq) - int(df_triggers['Time(Sample)'][t]-1),dtype=int) 
                segment_corr,segment_zygo =apply_segments(lenght_epoch_zygo,df_triggers['contraction'][t],df_triggers['muscle_type'][t][0],int(df_triggers['Time(Sample)'][t]-1))

                segments.loc[i] = [subject, int(block),  t + 1,segment_corr,segment_zygo] 
                i += 1 
        

                

/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_7109/74793735.py:24: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_trials_filtered_subj = df_trials_filtered_subj[df_trials_filtered['Nap_ID'] == int(block)]
/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_7109/74793735.py:24: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_trials_filtered_subj = df_trials_filtered_subj[df_trials_filtered['Nap_ID'] == int(block)]
/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_7109/74793735.py:24: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_trials_filtered_subj = df_trials_filtered_subj[df_trials_filtered['Nap_ID'] == int(block)]
/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_7109/74793735.py:24: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_trials_filtered_subj = df_trials_filtered_subj[df_trials_filtered['Nap_ID'] == int(

KeyboardInterrupt: 